# Sentinel-1 → gamma0 RTC VV/VH on a common grid, via ASF HyP3

Takes four Sentinel-1 acquisitions and an AOI, has ASF HyP3 produce
radiometrically terrain-corrected gamma0, then regrids everything onto one shared
grid and writes a GeoTIFF per date with two named bands (`gamma0_VV`,
`gamma0_VH`).

**Why HyP3 rather than a local SAR stack.** OTB has no terrain flattening and is
not packaged for Windows; ISCE2/ISCE3 are Linux-only and do not calibrate
radiometrically. HyP3 runs the processing on ASF's servers and returns exactly the
product asked for: gamma0 RTC, VV and VH, GeoTIFF, UTM.

**What "common grid" means here — and what it does not.** The alignment is purely
*geometric*: cell 9 derives one UTM lattice from the AOI alone, and cell 10
resamples every date onto it, so the same array index holds the same patch of
ground in all four files. Two things are deliberately *not* done:

- no **radiometric** coregistration — no cross-calibration, no histogram matching
  between dates. Each gamma0 keeps its own absolute calibration, so a difference
  between two dates is a real change on the ground rather than something a
  normalisation has flattened. That is what a time series needs;
- no **InSAR** coregistration — no phase, no sub-pixel refinement. Whatever
  misregistration HyP3's geocoding left, usually well under a pixel thanks to the
  precise orbits and the Copernicus DEM, stays.

## How to use it

**1. Install the dependencies.**

```
conda install -c conda-forge asf_search hyp3_sdk rasterio geopandas numpy
```

**2. Get a NASA Earthdata Login** (free, <https://urs.earthdata.nasa.gov>) and
store the credentials in a netrc file in your home directory — on Windows,
`C:/Users/<you>/.netrc`:

```
machine urs.earthdata.nasa.gov
login <username>
password <password>
```

`requests` finds that file on its own, by matching the host name. Nothing else to
configure.

**3. Provide the inputs**, in the parameters cell. Only the first two really need
your attention; everything else ships with a working default.

| Parameter | What to give |
| --- | --- |
| `SLC_PATHS` | The acquisitions to process, as local `.SAFE` / `.zip` paths **or** as bare granule names (`S1A_IW_SLC__1SDV_20241018T094629_..._5D7D`). They only say *which* acquisitions to process — ASF works on its own copy, nothing is uploaded. Supplying a local file additionally lets cell 4 check the coverage burst by burst instead of scene by scene. |
| `AOI` | The area of interest in lon/lat (EPSG:4326): inline WKT, or the path to a `.wkt` / `.geojson` file. It drives both the coverage check and the extent of the output rasters. |
| `MODE` | `"slc_scene"`, `"slc_burst"` or `"grd"` — see below. Default `"slc_scene"`. |
| `PIXEL_SIZE` | Output resolution in metres: 10, 20 or 30. Default 10. |
| `POLARISATIONS` | The bands to produce, in order. Default `["VV", "VH"]`. |
| `DOWNLOAD_DIR`, `OUT_DIR` | Where the raw products and the final GeoTIFFs land. Relative to this notebook by default; point them outside the repository if you can, the products are heavy. Keep them distinct between two runs — cell 10 searches the tree by date, not by job. |
| `JOB_NAME` | The label stamped on the jobs, and the way both you and cell 7 find them again. Built from the mode and the resolution by default. Changing it starts a clean slate: every granule is then submitted afresh, and paid for again. |

Everything else — the RTC options, the acquisition dates — is derived from these.

**4. Run cells 1 to 5 and read them.** Cell 4 must report `OK` for all
acquisitions; cell 5 lists the granules that will actually be submitted. Nothing
has been spent at this stage — both cells only query the free catalogue.

**5. Run cell 6** and compare the credit balance with the number of granules from
cell 5. This is the moment to drop to 20 or 30 m if the budget is tight.

**6. Run cell 7.** For each granule it either recovers the job already filed under
`JOB_NAME` or submits a new one — the only cell that can spend credits. Because
the check is per granule, three things all work: re-running it after a crash or a
restart recovers the batch without paying again, adding acquisitions to
`SLC_PATHS` submits only the new ones, and a failed job is retried. It is also how
you rebuild `batch` before resuming at cell 8.

**7. Run cell 8, then re-run it every few minutes.** It never blocks: it reports
the job statuses and, once none is left pending or running, downloads and
extracts. Processing takes minutes to a few hours, and nothing is lost if you
close the notebook meanwhile — cell 7 rebuilds the batch from `JOB_NAME`.
Downloading is safe to repeat: it costs bandwidth, not credits. Do not postpone
it too long, HyP3 deletes the products after about two weeks.

**8. Run cells 9 to 11.** Purely local. Change `PIXEL_SIZE` or the AOI and replay
them as often as you like, with no reprocessing and no cost.

Results land in `OUT_DIR`: one GeoTIFF per acquisition, bands `gamma0_VV` and
`gamma0_VH`, all sharing the same grid so they stack pixel to pixel.

**After a kernel restart**, run cells 2, 3, 5, 6 and 7 again to rebuild the state,
then carry on at cell 8. Nothing is resubmitted and nothing is paid twice.

## Three modes

Set by `MODE` in the parameters cell. All three end on the same output.

| | `"slc_scene"` | `"slc_burst"` | `"grd"` |
| --- | --- | --- | --- |
| source product | SLC | SLC, burst by burst | GRD |
| processed area | the whole IW scene | only the bursts touching the AOI | the whole scene |
| jobs | 1 per date | 1 per burst *and* per polarisation | 1 per date |
| relative cost | high | lowest | moderate |
| native resolution | ~5 × 20 m | ~5 × 20 m | ~20 m |
| sensible `PIXEL_SIZE` | 10 m | 10 m | 20 or 30 m |
| granule ids | the product names | `asf_search`, `BURST` | `asf_search`, `GRD_HD` |

**Which one to pick.** `slc_scene` is the default: the most direct route, since
the granules are simply the product names, with nothing to look up. It is also
the heaviest, since the whole scene is processed and downloaded whatever the size
of the AOI. `slc_burst` processes only what the AOI touches, so it is the
cheapest and the fastest once the burst granules exist for your acquisitions.
`grd` starts from an already detected, multi-looked product: the phase is gone,
which rules out any later interferometry, but that is irrelevant for gamma0
amplitude, and the source is some eight times lighter.

Burst granules carry ESA's *absolute* burst ids, unrelated to the product-local
numbering of `polygon_to_swaths_bursts`. Rather than converting between the two,
`slc_burst` asks `asf_search` which bursts intersect the AOI — and cell 4 keeps
our own selection as a cross-check.

## What each cell does

The **ASF** column separates free catalogue queries from the calls that consume
credits, so nothing is spent before everything checkable has been checked.

| # | What it does | ASF |
| --- | --- | --- |
| 1 | This overview. | — |
| 2 | Imports, and puts the sibling `polygon_to_swaths_bursts` folder on `sys.path`. | — |
| 3 | Every parameter: mode, the acquisitions, the AOI, directories, pixel size, job name, RTC options. | — |
| 4 | Coverage check: does each acquisition really reach the AOI, and through which swaths and bursts. Falls back to the scene footprint when the product is not available locally. | catalogue, free |
| 5 | Builds the list of granules to submit — the product names in `slc_scene`, one catalogue query per acquisition otherwise (`BURST` or `GRD_HD`). | catalogue, free |
| 6 | Connects to HyP3 through the netrc, prints the credit balance and the real `submit_rtc_job` signature. | yes |
| 7 | For each granule, recovers the job already filed under `JOB_NAME` or submits a new one. **The only cell that can spend credits.** | yes |
| 8 | Reports the job statuses without blocking, and downloads and extracts once they are all done. Re-run it until then. | yes |
| 9 | Computes the common output grid: AOI bounding box in UTM, snapped to the pixel size. | — |
| 10 | Regrids every downloaded raster onto that grid, merges the tiles of a same date, writes one 2-band GeoTIFF per acquisition. | — |
| 11 | Reports the outputs: grid of each file and share of valid pixels. | — |

In [1]:
# === 2. Imports ===
import inspect
import sys
import zipfile
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

import asf_search as asf
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import Resampling, reproject

# polygon_to_swaths_bursts is a sibling folder: make it importable
TOOLS = Path.cwd().parent / "polygon_to_swaths_bursts"
sys.path.insert(0, str(TOOLS))
from polygon_to_swaths_bursts import get_intersecting_bursts, parse_polygon

In [2]:
# === 3. Parameters: adapt to your data ===

# "slc_scene" : the whole IW scene, processed from the SLC product
# "slc_burst" : only the bursts touching the AOI — cheapest and fastest
# "grd"       : the whole scene, from the lighter GRD product. No phase and no
#               bursts, but plenty for gamma0 amplitude — and ~20 m native
#               resolution, so do not ask for 10 m output in this mode.
MODE = "slc_scene"

# The acquisitions to process, named by their SLC product. ASF works on its own
# copy — nothing is uploaded — so each entry may be either a local .SAFE / .zip
# path or just the bare granule name:
#     "S1B_IW_SLC__1SDV_20170804T215105_20170804T215131_006796_00BF5A_B333"
# Only the AOI coverage check in cell 4 opens the local files, and it falls back
# to the ASF catalogue for the entries it cannot find. slc_scene submits these
# names as granules; slc_burst and grd use their acquisition times to look up the
# matching burst or GRD granules in the catalogue.
SLC_PATHS = [
    r"C:\Users\guigu\Documents\pro_asus\vigisar\data\data_raw\zta6\S1A_IW_SLC__1SDV_20241018T094629_20241018T094656_056155_06DF67_5D7D.zip",
    r"C:\Users\guigu\Documents\pro_asus\vigisar\data\data_raw\zta6\S1A_IW_SLC__1SDV_20241030T094629_20241030T094656_056330_06E655_1096.zip",
    r"C:\Users\guigu\Documents\pro_asus\vigisar\data\data_raw\zta6\S1A_IW_SLC__1SDV_20250603T094621_20250603T094648_059480_076248_F6BE.zip",
    r"C:\Users\guigu\Documents\pro_asus\vigisar\data\data_raw\zta6\S1A_IW_SLC__1SDV_20250615T094620_20250615T094647_059655_07683D_E467.zip",
]

# Area of interest, lon/lat (EPSG:4326): inline WKT, or a WKT / GeoJSON file path
AOI = "POLYGON ((-61.443781662296 2.51616906357776, -61.1231301454919 2.51574934499795, -61.1234621729535 2.27171671481147, -61.444129126251 2.27224523654185, -61.443781662296 2.51616906357776))"

# Where the files land. A relative path resolves against this notebook's folder;
# an absolute one ("C:/data/hyp3") works just as well and is the better habit,
# since these products are heavy and do not belong in the repo. Both directories
# are created on the fly, missing parent folders included.
DOWNLOAD_DIR = "hyp3_downloads/test"   # raw HyP3 products: zips, then extracted
OUT_DIR = "output/test"                # the final 2-band GeoTIFFs

# Metres; HyP3 RTC accepts 10, 20 or 30. GRD resolves at ~20 m, so asking 10 m
# in grd mode oversamples without adding information — prefer 20 or 30 there.
PIXEL_SIZE = 10.0
POLARISATIONS = ["VV", "VH"]

# Free-text label stamped on every job, and the only handle to find them again
# later with hyp3.find_jobs(name=...). It is not unique server-side, so include
# what distinguishes one submission from another: two runs sharing a name come
# back mixed together.
JOB_NAME = f"zta6-{MODE}-{int(PIXEL_SIZE)}m"

# gamma0 + power is the standard pair for analysis: keep the linear scale here
# and convert to dB only for display. Burst jobs accept fewer options than the
# scene-level ones — check against the signature printed in cell 6.
RTC_OPTIONS = dict(
    radiometry="gamma0",
    scale="power",
    resolution=int(PIXEL_SIZE),
)
if MODE != "slc_burst":
    RTC_OPTIONS |= dict(
        dem_name="copernicus",
        speckle_filter=False,   # filtering is a downstream choice, keep raw data
        dem_matching=False,     # can degrade geolocation over flat or wet terrain
    )


def acquisition_window(slc_path):
    """(start, stop) datetimes read from a Sentinel-1 product name."""
    parts = Path(slc_path).stem.split("_")
    return (
        datetime.strptime(parts[5], "%Y%m%dT%H%M%S"),
        datetime.strptime(parts[6], "%Y%m%dT%H%M%S"),
    )


DATES = [acquisition_window(p)[0].strftime("%Y%m%d") for p in SLC_PATHS]

In [3]:
# === 4. Does the AOI really fall inside each acquisition? ===
# In slc_scene mode this is the only guard against paying to process a product
# that misses the AOI. In slc_burst and grd modes the catalogue query of cell 5
# selects by intersection anyway, so this is a cross-check. With a local product
# the check is burst by burst; without one it falls back to the scene footprint
# from the ASF catalogue — a free query, no download, which also confirms that
# the granule name exists.
aoi_geom = parse_polygon(AOI)

for slc in SLC_PATHS:
    name = Path(slc).stem

    if Path(slc).exists():
        _, summary = get_intersecting_bursts(slc, AOI, coarse=True)
        if summary:
            detail = ", ".join(f"{sw} {bursts}" for sw, bursts in sorted(summary.items()))
            print(f"OK      {name[:58]}\n        {detail}")
        else:
            print(f"NO DATA {name[:58]} — the AOI is outside this product")
        continue

    results = asf.granule_search([name])
    if not results:
        print(f"UNKNOWN {name[:58]} — no such granule in the ASF catalogue")
    elif any(parse_polygon(r.geometry).intersects(aoi_geom) for r in results):
        print(f"OK      {name[:58]}\n        AOI inside the scene footprint (catalogue)")
    else:
        print(f"NO DATA {name[:58]} — the AOI is outside this scene")

c:\Users\guigu\Documents\pro_asus\geo\polygon_to_swaths_bursts\polygon_to_swaths_bursts.py:265: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  footprints.buffer(coarse_margin) if coarse else footprints.geometry
c:\Users\guigu\Documents\pro_asus\geo\polygon_to_swaths_bursts\polygon_to_swaths_bursts.py:265: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  footprints.buffer(coarse_margin) if coarse else footprints.geometry


OK      S1A_IW_SLC__1SDV_20241018T094629_20241018T094656_056155_06
        IW3 [4, 5, 6]
OK      S1A_IW_SLC__1SDV_20241030T094629_20241030T094656_056330_06
        IW3 [4, 5, 6]
OK      S1A_IW_SLC__1SDV_20250603T094621_20250603T094648_059480_07
        IW3 [4, 5, 6]
OK      S1A_IW_SLC__1SDV_20250615T094620_20250615T094647_059655_07
        IW3 [4, 5, 6]


c:\Users\guigu\Documents\pro_asus\geo\polygon_to_swaths_bursts\polygon_to_swaths_bursts.py:265: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  footprints.buffer(coarse_margin) if coarse else footprints.geometry
c:\Users\guigu\Documents\pro_asus\geo\polygon_to_swaths_bursts\polygon_to_swaths_bursts.py:265: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  footprints.buffer(coarse_margin) if coarse else footprints.geometry


In [4]:
# === 5. The granules to submit ===
aoi_wkt = parse_polygon(AOI).wkt

if MODE == "slc_scene":
    # The ASF scene identifier is the product name without its extension
    GRANULES = [Path(p).stem for p in SLC_PATHS]
else:
    # Ask the catalogue which granules cover the AOI, one acquisition at a time.
    # A one-minute margin around the product times isolates that acquisition.
    level = "BURST" if MODE == "slc_burst" else "GRD_HD"
    # A burst granule holds a single polarisation; a GRD scene carries both
    pols = POLARISATIONS if MODE == "slc_burst" else ["VV+VH"]

    GRANULES = []
    for slc in SLC_PATHS:
        start, stop = acquisition_window(slc)
        results = asf.search(
            platform=asf.PLATFORM.SENTINEL1,
            processingLevel=level,
            intersectsWith=aoi_wkt,
            start=start - timedelta(minutes=1),
            end=stop + timedelta(minutes=1),
            polarization=pols,
        )
        found = sorted(r.properties["fileID"] for r in results)
        print(f"{Path(slc).stem[:52]}: {len(found)} {level} granule(s)")
        for granule in found:
            print("   ", granule)
        GRANULES += found

print(f"\n{len(GRANULES)} granules to submit in {MODE} mode")


4 granules to submit in slc_scene mode


In [5]:
# === 6. Connect to HyP3 and check what this version accepts ===
import hyp3_sdk as sdk

print("hyp3_sdk", sdk.__version__)

# Credentials are read from the netrc file in your home directory
# (C:/Users/<you>/.netrc, or _netrc — requests accepts either):
#
#     machine urs.earthdata.nasa.gov
#     login <earthdata username>
#     password <earthdata password>
#
# Called without arguments, HyP3 lets requests pick them up from there. Pass
# prompt="password" or prompt="token" instead to be asked interactively.
netrc = next(
    (p for p in (Path.home() / ".netrc", Path.home() / "_netrc") if p.exists()), None
)
if netrc is None:
    raise FileNotFoundError(
        f"no .netrc or _netrc found in {Path.home()} — create one as shown above, "
        'or switch to sdk.HyP3(prompt="password")'
    )
print("credentials from", netrc)

hyp3 = sdk.HyP3()

info = hyp3.my_info()
print("user:", info.get("user_id"), "| remaining credits:", info.get("remaining_credits"))

# Compare RTC_OPTIONS above with the keywords this version really accepts
print("\nsubmit_rtc_job", inspect.signature(hyp3.submit_rtc_job))

hyp3_sdk 7.7.8
credentials from C:\Users\guigu\.netrc
user: gbonlieu | remaining credits: 7580

submit_rtc_job (granule: str, name: str | None = None, dem_matching: bool = False, include_dem: bool = False, include_inc_map: bool = False, include_rgb: bool = False, include_scattering_area: bool = False, radiometry: Literal['sigma0', 'gamma0'] = 'gamma0', resolution: Literal[10, 20, 30] = 30, scale: Literal['amplitude', 'decibel', 'power'] = 'power', speckle_filter: bool = False, dem_name: Literal['copernicus'] = 'copernicus') -> hyp3_sdk.jobs.Batch


In [6]:
# === 7. Submit the missing jobs, recover the ones already filed ===
# The guard works granule by granule, so all three situations behave sensibly:
# re-running after a crash recovers everything, adding acquisitions to SLC_PATHS
# submits only the new ones, and nothing is ever paid for twice under the same
# JOB_NAME. Failed jobs are ignored, which retries them.
#
# HyP3 keeps job records long after the products expire, so the lookup is limited
# to the last two weeks: an older job cannot be downloaded any more anyway.
recent = datetime.now(timezone.utc) - timedelta(days=14)
existing = hyp3.find_jobs(name=JOB_NAME, start=recent)


def job_granules(job):
    """The granules one job was submitted for."""
    return job.job_parameters.get("granules", [])


wanted = set(GRANULES)
kept = [
    job for job in existing
    if job.status_code != "FAILED" and any(g in wanted for g in job_granules(job))
]
covered = {g for job in kept for g in job_granules(job)}

# Failing loudly here is much cheaper than silently resubmitting everything
if existing and not covered:
    raise RuntimeError(
        f"{len(existing)} job(s) found under {JOB_NAME!r} but their granules could "
        "not be read — refusing to resubmit and risk paying twice. Inspect "
        "existing[0].job_parameters and fix job_granules()."
    )

batch = sdk.Batch(kept)
for granule in GRANULES:
    if granule not in covered:
        batch += hyp3.submit_rtc_job(granule, name=JOB_NAME, **RTC_OPTIONS)

print(f"{JOB_NAME}: {len(kept)} recovered, {len(batch) - len(kept)} submitted")
for job in batch:
    print(f"    {job.status_code:9s} {job.job_id}")

zta6-slc_scene-10m: 4 recovered, 0 submitted
    SUCCEEDED a849dfb4-2f81-49da-a0e0-8272553dab89
    SUCCEEDED 1db7fed0-c421-490a-b535-30a41d7d247e
    SUCCEEDED 35bc2cdf-8477-4f4c-8bdd-9be8f68ea421
    SUCCEEDED 901fb7ca-a48b-4b7c-9aee-f462fb838c77


In [7]:
# === 8. Check the jobs, and download once they are done ===
# Deliberately NOT blocking. hyp3.watch() would hold the kernel for hours, and
# interrupting a blocked kernel is what kills it. Re-run this cell every few
# minutes instead: it reports the statuses, and downloads as soon as all the
# jobs have left PENDING and RUNNING.
#
# HyP3 publishes no progress percentage — only PENDING (queued), RUNNING
# (a worker has it) and SUCCEEDED / FAILED. The elapsed time below is the only
# usable proxy: a whole-scene RTC usually lands within the hour.
batch = hyp3.refresh(batch)
now = datetime.now(timezone.utc)

print(" | ".join(f"{code}: {n}"
                 for code, n in sorted(Counter(j.status_code for j in batch).items())))
for job in batch:
    requested = getattr(job, "request_time", None)
    age = f"{(now - requested).total_seconds() / 60:4.0f} min" if requested else "  ? min"
    print(f"    {job.status_code:9s} {age}  {job.job_id}")

waiting = [job for job in batch if job.status_code in ("PENDING", "RUNNING")]
if waiting:
    print(f"\n{len(waiting)} job(s) still going — re-run this cell in a few minutes")
else:
    failed = [job for job in batch if job.status_code == "FAILED"]
    if failed:
        print(f"\n{len(failed)} job(s) FAILED — re-run cell 7 to retry them")

    succeeded = sdk.Batch([j for j in batch if j.status_code == "SUCCEEDED"])
    download_dir = Path(DOWNLOAD_DIR)
    download_dir.mkdir(parents=True, exist_ok=True)
    zips = succeeded.download_files(location=download_dir)

    for archive in zips:
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(download_dir)
        print("extracted:", Path(archive).name)

SUCCEEDED: 4
    SUCCEEDED   73 min  a849dfb4-2f81-49da-a0e0-8272553dab89
    SUCCEEDED   73 min  1db7fed0-c421-490a-b535-30a41d7d247e
    SUCCEEDED   73 min  35bc2cdf-8477-4f4c-8bdd-9be8f68ea421
    SUCCEEDED   73 min  901fb7ca-a48b-4b7c-9aee-f462fb838c77


S1A_IW_20250615T094620_DVP_RTC10_G_gpuned_8A66.zip: 100%|██████████| 3.80G/3.80G [03:53<00:00, 17.5MB/s]
S1A_IW_20250603T094621_DVP_RTC10_G_gpuned_EFD7.zip: 100%|██████████| 3.80G/3.80G [04:13<00:00, 16.1MB/s]
S1A_IW_20241030T094629_DVP_RTC10_G_gpuned_45BA.zip: 100%|██████████| 3.82G/3.82G [04:41<00:00, 14.6MB/s]
S1A_IW_20241018T094629_DVP_RTC10_G_gpuned_5FF0.zip: 100%|██████████| 3.81G/3.81G [05:03<00:00, 13.5MB/s]
100%|██████████| 4/4 [17:55<00:00, 268.99s/it]


extracted: S1A_IW_20250615T094620_DVP_RTC10_G_gpuned_8A66.zip
extracted: S1A_IW_20250603T094621_DVP_RTC10_G_gpuned_EFD7.zip
extracted: S1A_IW_20241030T094629_DVP_RTC10_G_gpuned_45BA.zip
extracted: S1A_IW_20241018T094629_DVP_RTC10_G_gpuned_5FF0.zip


In [8]:
# === 9. The common output grid: AOI bbox in UTM, snapped to PIXEL_SIZE ===
aoi_gs = gpd.GeoSeries([parse_polygon(AOI)], crs="EPSG:4326")
UTM_CRS = aoi_gs.estimate_utm_crs()
minx, miny, maxx, maxy = aoi_gs.to_crs(UTM_CRS).total_bounds

# The grid comes from the AOI alone, never from the images: that is what makes
# every date land on exactly the same pixel centres.
#
# Snapping the origin to a round multiple of PIXEL_SIZE is a separate matter. It
# turns the grid into a canonical lattice attached to (CRS, pixel size) rather
# than to this particular AOI, which buys three things:
#   - editing the AOI later moves the extent by whole pixels instead of sliding
#     the lattice, so new outputs still stack on the old ones;
#   - neighbouring AOIs snapped the same way share the lattice and mosaic
#     without resampling;
#   - other products already on round grids — Sentinel-2 tiles at 10 m in UTM,
#     notably — overlay pixel to pixel, which matters for radar/optical fusion.
ULX = np.floor(minx / PIXEL_SIZE) * PIXEL_SIZE
ULY = np.ceil(maxy / PIXEL_SIZE) * PIXEL_SIZE
SIZE_X = int(np.ceil((maxx - ULX) / PIXEL_SIZE))
SIZE_Y = int(np.ceil((ULY - miny) / PIXEL_SIZE))
TRANSFORM = from_origin(ULX, ULY, PIXEL_SIZE, PIXEL_SIZE)

print(f"{UTM_CRS.name} (EPSG:{UTM_CRS.to_epsg()})")
print(f"{SIZE_X} x {SIZE_Y} px at {PIXEL_SIZE} m, upper-left ({ULX}, {ULY})")

WGS 84 / UTM zone 20N (EPSG:32620)
3568 x 2700 px at 10.0 m, upper-left (673010.0, 278220.0)


In [9]:
# === 10. Regrid, merge and write one 2-band GeoTIFF per acquisition ===
def find_rtc_bands(date, pol, search_dir):
    """HyP3 RTC GeoTIFFs for one date and polarisation.

    A single file in slc_scene and grd modes, one per burst in slc_burst mode —
    HyP3 names its outputs after the input granule, so all carry the date.
    """
    matches = [
        p for p in Path(search_dir).rglob(f"*_{pol}.tif")
        if date in p.name and "rgb" not in p.name.lower()
    ]
    if not matches:
        raise FileNotFoundError(f"no {pol} RTC file for {date} under {search_dir}")
    return sorted(matches)


def on_common_grid(src_path):
    """Resample one RTC GeoTIFF onto the shared grid. Nodata becomes NaN."""
    target = np.full((SIZE_Y, SIZE_X), np.nan, dtype="float32")
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=target,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=TRANSFORM,
            dst_crs=UTM_CRS,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return target


def merge_on_grid(paths):
    """Regrid several tiles and merge them: average where they overlap."""
    total = np.zeros((SIZE_Y, SIZE_X), dtype="float64")
    count = np.zeros((SIZE_Y, SIZE_X), dtype="uint16")
    for path in paths:
        values = on_common_grid(path)
        valid = np.isfinite(values) & (values > 0)
        total[valid] += values[valid]
        count[valid] += 1
    merged = np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)
    return merged.astype("float32")


outputs = []
for slc, date in zip(SLC_PATHS, DATES):
    bands = []
    for pol in POLARISATIONS:
        tiles = find_rtc_bands(date, pol, DOWNLOAD_DIR)
        print(f"{date} {pol}: {len(tiles)} tile(s)")
        bands.append(merge_on_grid(tiles))

    out_path = Path(OUT_DIR) / f"{Path(slc).stem}_gamma0.tif"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        out_path, "w", driver="GTiff",
        height=SIZE_Y, width=SIZE_X, count=len(POLARISATIONS),
        dtype="float32", crs=UTM_CRS, transform=TRANSFORM, nodata=np.nan,
        compress="deflate", tiled=True,
    ) as dst:
        for index, (pol, band) in enumerate(zip(POLARISATIONS, bands), start=1):
            dst.write(band, index)
            dst.set_band_description(index, f"gamma0_{pol}")

    print("written:", out_path)
    outputs.append(out_path)

20241018 VV: 1 tile(s)
20241018 VH: 1 tile(s)
written: output\test\S1A_IW_SLC__1SDV_20241018T094629_20241018T094656_056155_06DF67_5D7D_gamma0.tif
20241030 VV: 1 tile(s)
20241030 VH: 1 tile(s)
written: output\test\S1A_IW_SLC__1SDV_20241030T094629_20241030T094656_056330_06E655_1096_gamma0.tif
20250603 VV: 1 tile(s)
20250603 VH: 1 tile(s)
written: output\test\S1A_IW_SLC__1SDV_20250603T094621_20250603T094648_059480_076248_F6BE_gamma0.tif
20250615 VV: 1 tile(s)
20250615 VH: 1 tile(s)
written: output\test\S1A_IW_SLC__1SDV_20250615T094620_20250615T094647_059655_07683D_E467_gamma0.tif


In [10]:
# === 11. Check the outputs: same grid everywhere, enough valid pixels ===
for path in outputs:
    with rasterio.open(path) as src:
        print(path.name)
        print(f"    {src.crs}, {src.width}x{src.height} px, "
              f"pixel {src.transform.a:g} m")
        for index in range(1, src.count + 1):
            data = src.read(index)
            valid = np.isfinite(data) & (data > 0)
            print(f"    {src.descriptions[index - 1]}: "
                  f"{valid.sum() / data.size:.0%} valid pixels")

S1A_IW_SLC__1SDV_20241018T094629_20241018T094656_056155_06DF67_5D7D_gamma0.tif
    EPSG:32620, 3568x2700 px, pixel 10 m
    gamma0_VV: 100% valid pixels
    gamma0_VH: 100% valid pixels
S1A_IW_SLC__1SDV_20241030T094629_20241030T094656_056330_06E655_1096_gamma0.tif
    EPSG:32620, 3568x2700 px, pixel 10 m
    gamma0_VV: 100% valid pixels
    gamma0_VH: 100% valid pixels
S1A_IW_SLC__1SDV_20250603T094621_20250603T094648_059480_076248_F6BE_gamma0.tif
    EPSG:32620, 3568x2700 px, pixel 10 m
    gamma0_VV: 100% valid pixels
    gamma0_VH: 100% valid pixels
S1A_IW_SLC__1SDV_20250615T094620_20250615T094647_059655_07683D_E467_gamma0.tif
    EPSG:32620, 3568x2700 px, pixel 10 m
    gamma0_VV: 100% valid pixels
    gamma0_VH: 100% valid pixels
